# 37 — MLL23: embed a second external peripheral-blood source (P1-8, RQ4)

MLL23 (Scientific Data 12:1773, 2025; Zenodo DOI 10.5281/zenodo.14277609) holds 41,621 single-cell images from the Munich
Leukemia Laboratory, in 18 classes, including **8,606 myeloblasts**. Images are 288 x 288 px (25 x 25 um), Pappenheim stain,
Metafer scanner. Every image is embedded with the SAME frozen encoder and preprocessing as `03_embeddings_colab.ipynb`.
The data descriptor does not state whether patient identifiers are released; if they are not, uncertainty can only be
bootstrapped over images (as for RV-PBS), which this notebook records.

**Status: written 2026-10-19 without network access to Zenodo, so it has NOT been run.** Check the dataset licence on the
Zenodo record before using the data in the paper.
**Runtime:** T4 GPU. Runtime > Run all. Download: 15 of the 18 ZIP files, about 8.6 GB (20-60 min); embedding about 10 min.
**Licence (checked 2026-10-20 on the Zenodo page): CC BY 4.0.** Record title: "A large publicly available single-cell peripheral blood dataset (MLL23)".

In [ ]:
# 0. Setup
!pip -q install timm==1.0.* 2>/dev/null
import os, json, time, zipfile, hashlib, platform, requests
import numpy as np, pandas as pd, torch, timm
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/certified-adaptive-pbs/mll23_v1'; LOCAL = '/content/mll23'
os.makedirs(OUT, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

In [ ]:
# 1. Download the class ZIPs from Zenodo (streamed to disk) and record the licence and checksums.
# Skipped, because they never enter a specimen (see scripts/38): promyelocyte_atypical (blast equivalents under WHO 2022 /
# ICC 2022, which our labels do not count), smudge_cell and normoblast (not counted in a leukocyte differential).
SKIP = {'promyelocyte_atypical.zip', 'smudge_cell.zip', 'normoblast.zip'}
rec = requests.get('https://zenodo.org/api/records/14277609', timeout=60).json()
meta = dict(title=rec['metadata'].get('title'), license=rec['metadata'].get('license'), version=rec['metadata'].get('version'),
            doi=rec.get('doi'), skipped=sorted(SKIP), files=[])
print(meta['title'], '| licence:', meta['license'])
for f in rec['files']:
    name = f['key']; meta['files'].append(dict(name=name, size=f['size'], checksum=f.get('checksum')))
    if name in SKIP: print('skip', name); continue
    dst = f'{LOCAL}/{name}'; t0 = time.time()
    if not os.path.isdir(f'{LOCAL}/img/{name[:-4]}'):
        with requests.get(f['links']['self'], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(dst, 'wb') as fh:
                for chunk in r.iter_content(chunk_size=1 << 22): fh.write(chunk)
        if f.get('checksum', '').startswith('md5:'):
            h = hashlib.md5()
            with open(dst, 'rb') as fh:
                for chunk in iter(lambda: fh.read(1 << 22), b''): h.update(chunk)
            assert h.hexdigest() == f['checksum'][4:], f'checksum mismatch for {name}'
        zipfile.ZipFile(dst).extractall(f'{LOCAL}/img/{name[:-4]}'); os.remove(dst)    # free disk space
    print(name, round(f['size'] / 1e6, 1), 'MB', f'{time.time()-t0:.0f}s', flush=True)


In [ ]:
# 2. Index images; class = ZIP name. Keep the path inside the ZIP (it may encode a patient or sample identifier).
rows = []
for cls in sorted(os.listdir(f'{LOCAL}/img')):
    for root, _, fs in os.walk(f'{LOCAL}/img/{cls}'):
        for fn in fs:
            if fn.lower().endswith(('.tif', '.tiff', '.png', '.jpg', '.jpeg')):
                full = os.path.join(root, fn); rows.append(dict(cls=cls, file=full, relpath=os.path.relpath(full, f'{LOCAL}/img/{cls}'), image=fn))
M = pd.DataFrame(rows).sort_values(['cls', 'relpath']).reset_index(drop=True); print(len(M), 'images'); print(M.cls.value_counts())
print('example paths:', M.relpath.head(5).tolist())
other = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.lower().endswith(('.csv', '.xlsx', '.json', '.txt'))]
print('metadata files in the release (patient identifiers?):', other[:20])


In [ ]:
# 3. Embed with the encoder of notebook 03 (DINOv2 ViT-S/14, 224 px)
from PIL import Image
import torchvision.transforms.functional as TF
SEED = 20260919; torch.manual_seed(SEED)
MODEL_NAME, IMG_SIZE = 'vit_small_patch14_dinov2.lvd142m', 224
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0, img_size=IMG_SIZE).eval().to(device)
cfg = timm.data.resolve_data_config({}, model=model); MEAN, STD = cfg['mean'], cfg['std']
class DS(torch.utils.data.Dataset):
    def __len__(self): return len(M)
    def __getitem__(self, i):
        im = Image.open(M.file.iloc[i]).convert('RGB')
        return TF.normalize(TF.to_tensor(TF.resize(im, [IMG_SIZE, IMG_SIZE], antialias=True)), MEAN, STD), i
dl = torch.utils.data.DataLoader(DS(), batch_size=128, num_workers=2)
E = np.zeros((len(M), model.num_features), np.float32); t0 = time.time()
with torch.inference_mode():
    for b, (x, i) in enumerate(dl):
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
            E[i.numpy()] = model(x.to(device)).float().cpu().numpy()
        if b % 50 == 0: print(b, len(dl), f'{time.time()-t0:.0f}s')

In [ ]:
# 4. Save
np.save(f'{OUT}/mll23_embeddings_dinov2_vits14.npy', E.astype(np.float16))
M[['cls', 'relpath', 'image']].to_csv(f'{OUT}/mll23_cells.csv', index=False)
json.dump(dict(notebook='37_mll23_embeddings_colab.ipynb', model=MODEL_NAME, img_size=IMG_SIZE, n=len(M), zenodo=meta,
               torch=torch.__version__, timm=timm.__version__, python=platform.python_version(),
               created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())), open(f'{OUT}/config.json', 'w'), indent=1)
print('saved to', OUT)

## After it finishes
Download `mll23_v1/` into the repo at `data/mll23/`, then run `python scripts/38_mll23_shift.py`.